In [1]:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load the latest version
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
    "IMDB Dataset.csv",
    # kagglehub saves this file still zipped under the .csv name, so pandas has to unzip it
    pandas_kwargs={"compression": "zip"},
)

df.head(20)


/Users/heatherfrantisak/github.com/heathernfran/sentiment-analysis/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [2]:
df.review = df.review.str.replace(r"<[^>]+>", " ", regex=True)

df.review.str.contains(r"<[^>]+>").sum()

np.int64(0)

In [3]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df, test_size=0.3, random_state=100, stratify=df["sentiment"]
)

train

,review,sentiment
7059,As long as you can suffer it! If you like watc...,negative
27143,"I like Tom Hanks, and he is one of few actors ...",positive
28124,A documentary about a nomadic tribe in Tibet g...,positive
25720,"Not as bad as some are making it out to be, th...",negative
400,What can you say about the film White Fire. Am...,positive
...,...,...
48441,I wanted to see Valentine ever since I saw tha...,positive
15023,For weeks I had been looking forward to seeing...,negative
25029,This show is what happened to The Screen Saver...,negative
33821,"I actually went to see this film in a theater,...",negative


In [4]:
# Tokenize words
from spacy.lang.en import English
from spacy.tokenizer import Tokenizer

nlp_en = English()
tokenizer = Tokenizer(nlp_en.vocab)

In [5]:
# Remove stop words
from spacy.lang.en.stop_words import STOP_WORDS as S

print(len(S))

for stopword in S:
    print(stopword)

326
us
about
unless
're
your
less
'll
nine
does
whatever
indeed
hers
namely
until
rather
show
from
mostly
once
wherein
really
ca
forty
do
them
using
herself
because
will
’d
nowhere
regarding
front
whereupon
well
themselves
re
were
itself
another
anything
thru
‘m
five
something
an
always
put
due
therefore
keep
those
neither
make
perhaps
further
full
around
part
with
somehow
to
down
n't
towards
has
three
beside
anyway
did
all
somewhere
upon
during
if
toward
hence
seems
you
she
have
amount
take
get
across
had
such
amongst
’m
nor
the
very
beforehand
twenty
this
beyond
what
'd
who
he
as
often
name
one
hereupon
their
everyone
every
not
same
might
other
whence
though
enough
via
however
besides
therein
hundred
when
becoming
become
some
various
move
otherwise
fifteen
nobody
empty
go
four
n’t
nevertheless
being
since
more
‘re
‘ve
against
must
six
there
i
elsewhere
bottom
no
himself
sometime
between
others
latterly
least
whoever
any
never
a
last
afterwards
former
’s
our
seem
two
under
but
thereaf

In [6]:
keep_stop_words = {
    "not",
    "no",
    "never",
    "nor",
    "n't",
    "n‘t",
    "n’t",
    "nothing",
    "none",
    "neither",
    "but",
    "very",
    "too",
    "least",
    "against",
    "without",
    "cannot",
}

In [7]:
# Lemmatize words
nlp_en.add_pipe("lemmatizer")

In [8]:
# Assign part-of-speech tags
nlp_en.add_pipe("tagger")

In [9]:
print(nlp_en.pipeline)
print(nlp_en.pipe_names)

[('lemmatizer', <spacy.lang.en.lemmatizer.EnglishLemmatizer object at 0x11dcf9190>), ('tagger', <spacy.pipeline.tagger.Tagger object at 0x11dd0ca70>)]
['lemmatizer', 'tagger']


In [10]:
# Preprocessing Pipeline
import spacy

nlp = spacy.load("en_core_web_sm")

tokens = [
    [
        r.lemma_
        for r in review
        if not r.is_punct
        and not r.is_space
        and (not r.is_stop or r.lower_ in keep_stop_words)
    ]
    for review in nlp.pipe(train.review.head(), disable=["parser", "ner"])
]

for review_tokens in tokens:
    print(review_tokens)

['long', 'suffer', 'like', 'watch', 'people', 'wake', 'get', 'get', 'dress', 'have', 'shower', 'prepare', 'dinner', 'watch', 'have', 'sex', 'dark', 'go', 'bed', 'sleep', 'like', 'tacky', 'flat', 'narrow', 'bedroom', 'kitchen', 'long', 'minute', 'silence', 'like', 'get', 'bore', 'hour', 'feel', 'thrill', 'real', 'intimate', 'false', 'art', 'like', 'but', 'not', 'try', 'good', 'movie', 'thousand', 'long', 'but', 'want', 'stay', 'german', 'movie', 'get', 'award', 'Torino', 'gay', 'film', 'festival', 'italian', 'journalist', 'not', 'understand', 'jury', 'take', 'bad', 'decision', 'festival', 'present', 'lot', 'talented', 'movie', 'maybe', 'nice', 'German', 'not', 'award', 'life', 'but', 'excellent', 'but', 'not', 'gay', 'maybe', 'question', 'fashion', 'german', 'no', 'matter', 'maybe', 'hustler', 'glance', 'director']
['like', 'Tom', 'Hanks', 'actor', 'draw', 'theatre', 'regardless', 'misgiving', 'concern', 'film', 'worry', 'Mr.', 'Hanks', 'return', 'light', 'comedy', 'arena', 'transition'